In [1]:
import numpy as np
from skimage import filters
import dask
import dask.array
from dask.distributed import  Client, LocalCluster

memlim = '750GB' #3/4 of available RAM on machine

cluster = LocalCluster(memory_limit = memlim, n_workers=1)

client = Client(cluster) # code runs just fine without calling the LocalCluster, i.e. on the threaded? scheduler

def Gaussian_Blur_space(da, sigma):      
    deptharray = np.ones(da.ndim)+4*sigma
    deptharray[-1] = 0
    sigmas = np.ones(deptharray.shape)*sigma
    deptharray = tuple(np.min([deptharray, da.shape], axis=0))

    sigmas[-1] = 0
    G = da.map_overlap(filters.gaussian, depth=deptharray, boundary='nearest', sigma = sigmas, preserve_range=True)
    return G

shp = (750, 108, 2016, 53) # typical shape of image data: (750, 108, 2016, 53)

# im = (np.random.rand(*shp)*10000).astype(np.uint16)

# da = dask.array.from_array(im, chunks='auto')
da = dask.array.random.randint(1, 16000, shp)

In [2]:
da

dask.array<randint, shape=(750, 108, 2016, 53), dtype=int64, chunksize=(68, 68, 68, 53), chunktype=numpy.ndarray>

In [3]:
blur = Gaussian_Blur_space(da, 1)

blur = blur.compute()

2023-07-26 12:09:51,432 - distributed.utils_perf - WARNING - full garbage collections took 24% CPU time recently (threshold: 10%)
2023-07-26 12:09:53,188 - distributed.utils_perf - WARNING - full garbage collections took 24% CPU time recently (threshold: 10%)
2023-07-26 12:09:56,349 - distributed.utils_perf - WARNING - full garbage collections took 24% CPU time recently (threshold: 10%)
2023-07-26 12:10:03,049 - distributed.utils_perf - WARNING - full garbage collections took 24% CPU time recently (threshold: 10%)


In [14]:
blur

dask.array<_trim, shape=(250, 100, 400, 50), dtype=float64, chunksize=(115, 100, 115, 50), chunktype=numpy.ndarray>

In [15]:
blur = blur.compute()

/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/distributed/client.py:3141: UserWarning: Sending large graph of size 6.99 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
2023-07-26 11:29:12,673 - distributed.protocol.core - CRITICAL - Failed to Serialize
Traceback (most recent call last):
  File "/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/distributed/protocol/core.py", line 109, in dumps
    frames[0] = msgpack.dumps(msg, default=_encode_default, use_bin_type=True)
  File "/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/msgpack/__init__.py", line 38, in packb
    return Packer(**kwargs).pack(o)
  File "msgpack/_packer.pyx", line 294, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 300, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 297, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 264, in msgpack._cmsgpack.Packer._pack
  

CancelledError: ('_trim-f29bbfe41246458b3b3430ad0e74d59d', 4, 0, 5, 0)

In [15]:
blur = blur.compute() #warning with shp = (250, 100, 400, 50)

/mpc/homes/fische_r/miniconda3/lib/python3.10/site-packages/distributed/client.py:3141: UserWarning: Sending large graph of size 0.93 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
